# PCU-HYBRID-OBJECTIVE-001 — Narrow joint association/readout diagnostic

Engineering-only one-knob experiment. Freeze the published L7/K64 objective-alignment condition and add exactly one term: `L = L_rank + 0.25 * L_CE`, where `L_CE` uses the original answer-token task-sequence encoding.

Success requires **both** A_eval 16-way ranking accuracy >= 80% and A_eval greedy exact >= 80%. Formal seeds are never executed.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys

BRANCH = 'codex/pcu-composability-kill-001'
REPO = Path('/kaggle/working/mini-cells')
BASELINE = REPO / 'artifacts/research/pcu-objective-alignment-001/engineering/26090501-l7-k64-ranking'
OUT = REPO / 'artifacts/research/pcu-hybrid-objective-001/engineering/26090501-l7-k64-rank-plus-ce025'
FORMAL_SEEDS = (26090511, 26090512, 26090513)
REQUIRED_TRANSFORMERS = '5.16.1'
os.environ.setdefault('HF_HOME', '/kaggle/working/hf-cache')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')

def run(cmd, *, env=None, capture=False):
    cmd = [str(x) for x in cmd]
    print('+', ' '.join(cmd))
    result = subprocess.run(cmd, check=True, env=env, text=True, capture_output=capture)
    return result.stdout.strip() if capture else ''

if not REPO.exists():
    run(['git', 'clone', '--branch', BRANCH, 'https://github.com/ArcheLabs/mini-cells.git', REPO])
os.chdir(REPO)
run(['git', 'fetch', 'origin'])
run(['git', 'checkout', BRANCH])
run(['git', 'pull', '--ff-only', 'origin', BRANCH])
run([sys.executable, '-m', 'pip', 'install', '-e', '.[dev]'])
run([sys.executable, '-m', 'pip', 'install', f'transformers=={REQUIRED_TRANSFORMERS}', 'huggingface_hub>=0.36,<2.0', 'safetensors>=0.4', 'accelerate>=1.0'])

import torch, transformers
assert transformers.__version__ == REQUIRED_TRANSFORMERS
assert torch.cuda.is_available() and torch.cuda.device_count() >= 1
print(json.dumps({
    'commit': run(['git', 'rev-parse', 'HEAD'], capture=True),
    'tree': run(['git', 'rev-parse', 'HEAD^{tree}'], capture=True),
    'gpu0': torch.cuda.get_device_name(0),
    'transformers': transformers.__version__,
}, indent=2))


In [ ]:
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login
secrets = UserSecretsClient()
hf_token = secrets.get_secret('HF_TOKEN')
github_token = secrets.get_secret('GITHUB_TOKEN')
assert hf_token and github_token
os.environ['HF_TOKEN'] = hf_token
os.environ['HUGGING_FACE_HUB_TOKEN'] = hf_token
os.environ['GITHUB_TOKEN'] = github_token
login(token=hf_token, add_to_git_credential=False)
print('Secrets loaded; values not printed.')


In [ ]:
SEED_REGISTRY = REPO / 'research/formal_seed_registry.json'
def formal_states():
    payload = json.loads(SEED_REGISTRY.read_text())
    return {int(row['seed']): row['state'] for row in payload['seeds']}
expected = {seed: 'RESERVED_UNTOUCHED' for seed in FORMAL_SEEDS}
assert formal_states() == expected
assert run(['git', 'hash-object', SEED_REGISTRY], capture=True) == '71a3015a7d54e795538b3aa6750860f0b9168cb3'
print(json.dumps({'formal_seed_states': formal_states()}, indent=2))


In [ ]:
required_baseline = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json', 'PAIRED_CE_K64.json']
missing = [name for name in required_baseline if not (BASELINE / name).is_file()]
assert not missing, f'Published objective-alignment baseline missing: {missing}'
baseline_decision = json.loads((BASELINE / 'DECISION.json').read_text())
paired_ce = json.loads((BASELINE / 'PAIRED_CE_K64.json').read_text())
assert baseline_decision['status'] == 'ASSOCIATION_LEARNED_GENERATION_UNRESOLVED'
assert abs(baseline_decision['ranking_eval_accuracy'] - 0.8203125) < 1e-12
assert abs(baseline_decision['direct_accuracy'] - 0.0) < 1e-12
assert abs(paired_ce['direct_accuracy'] - 0.265625) < 1e-12
assert len(paired_ce['selected_cells']) == 64
remote_path = 'artifacts/research/pcu-objective-alignment-001/engineering/26090501-l7-k64-ranking/DECISION.json'
assert subprocess.run(['git', 'show', f'origin/{BRANCH}:{remote_path}'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL).returncode == 0
print(json.dumps({
    'ranking_eval_baseline': baseline_decision['ranking_eval_accuracy'],
    'ranking_greedy_baseline': baseline_decision['direct_accuracy'],
    'ce_greedy_baseline': paired_ce['direct_accuracy'],
    'selected_k': len(paired_ce['selected_cells']),
}, indent=2))


In [ ]:
test_env = os.environ.copy()
test_env['PYTHONPATH'] = str(REPO / 'src')
test_env['PYTEST_DISABLE_PLUGIN_AUTOLOAD'] = '1'
run([sys.executable, '-m', 'pytest', '-q', 'tests/research/05-pcu-kill-001'], env=test_env)
run([sys.executable, '-m', 'compileall', '-q', 'src/minicells/pcu_kill_001', 'scripts/research'])
print('PCU hybrid-objective test/compile gate: PASS')


In [ ]:
if OUT.exists():
    existing = sorted(p.name for p in OUT.glob('*.json'))
    assert not existing, f'Hybrid output already exists; inspect before rerun: {existing}'
run([
    sys.executable, 'scripts/research/run_pcu_hybrid_objective_001.py',
    '--seed', '26090501',
    '--device', 'cuda:0',
    '--baseline', BASELINE,
    '--out', OUT,
])


In [ ]:
required = ['RUN_IDENTITY.json', 'DESIGN.json', 'RESULT.json', 'DECISION.json']
missing = [name for name in required if not (OUT / name).is_file()]
assert not missing, missing
decision = json.loads((OUT / 'DECISION.json').read_text())
result = json.loads((OUT / 'RESULT.json').read_text())
assert decision['valid_run'] is True
assert decision['formal_execution_not_started'] is True
assert decision['selected_cells_exact_baseline_match'] is True
assert abs(decision['ce_weight'] - 0.25) < 1e-12
assert result['selected_cells'] == paired_ce['selected_cells']
assert formal_states() == expected
print(json.dumps({
    'status': decision['status'],
    'ce_weight': decision['ce_weight'],
    'ranking_train_accuracy': decision['ranking_train_accuracy'],
    'ranking_eval_accuracy': decision['ranking_eval_accuracy'],
    'direct_accuracy': decision['direct_accuracy'],
    'ranking_passes': decision['ranking_passes'],
    'direct_passes': decision['direct_passes'],
    'final_ranking_loss': result['training']['final_ranking_loss'],
    'final_ce_loss': result['training']['final_ce_loss'],
    'final_hybrid_loss': result['training']['final_hybrid_loss'],
}, indent=2))


In [ ]:
run([sys.executable, 'scripts/research/publish_pcu_hybrid_objective_001.py', '--branch', BRANCH])
assert formal_states() == expected
print(json.dumps({'published': True, 'formal_seed_states': formal_states()}, indent=2))
